# TP2 — Inverse Stage  
## Parameter Identification via Physics-Informed Neural Networks

This notebook implements the **inverse stage** of **Test Problem 2 (TP2)**.  
The objective is to **identify the unknown physical parameters** of a parametrized Poisson equation defined on a complex 3D geometry, by exploiting **Physics-Informed Neural Networks (PINNs)** and simulated IoT-like measurements.

This stage corresponds to the **Inverse Problem Submodule** of the Inference Engine described in the paper.

In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP2_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP2_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"
load_data_bound = ABS_PATH + "./files/data.csv"

load_initial_points = ABS_PATH + "files/initial.csv"
load_boundary_points = ABS_PATH + "files/gamma.csv"
load_collocation_points = ABS_PATH + "files/omega.csv"

fig_dir = "figures/"
loss_fig = ABS_PATH + fig_dir + "loss.png"
param_fig = ABS_PATH + fig_dir + "param.png"

params_save = ABS_PATH + "files/pred_parameters.csv"

Import of packages

In [ ]:
# Import
import torch
import pandas as pd
import matplotlib.pyplot as plt

from modelaquisition.bl2pina import Blend2Pina
from pina.condition import Condition
from pina.solvers.pinns import RBAPINN
from pina.equation import SystemEquation
from pina.callbacks import MetricTracker
from pina.geometry import CartesianDomain
from pina.model import ResidualFeedForward
from pina.operators import laplacian, grad
from pina import LabelTensor, Trainer, Plotter
from pytorch_lightning.callbacks import Callback, StochasticWeightAveraging
from pina.problem import SpatialProblem, InverseProblem, TimeDependentProblem

Set the double precision

In [ ]:
torch.set_default_dtype(torch.float64)

Definition of useful variables

In [ ]:
# Network variables
lear_rate = 5e-4
swa_lr = 5e-5
decay_rt = 1e-8

# Solver variables
epochs=10_000
batch=None
acc_str = 'gpu'

ipt_var = 4
out_var = 2
lay = 2
neur = 400

# Num points
int_points = 1_000
bound_points = 400
init_points = 400

## Geometry management and loading of data

Here the notebook focuses on the interaction between the preprocessed Blender 3D model (See notebook *00_ProblemSettings.ipynb*) with the PINA Geometry module.
In addition. in this fase the simulated data is loaded.

In [ ]:
column = Blend2Pina(LOAD_MODEL + model_name)

column_int = column.intern(time_interval=[0, 1])
column_bound = column.boundary(time_interval=[0, 1])
column_initial = column.intern(time_interval=[0, 1])

In [ ]:
df = pd.read_csv(load_data_bound, sep=";", index_col=0)

input_pts = df.iloc[:, :4].values
output_pts = df.iloc[:, 4:].values

input_pts = LabelTensor(
    x=torch.tensor(input_pts, dtype=torch.float64),
    labels=['x', 'y', 'z', 't']
)
output_pts = LabelTensor(
    x=torch.reshape(torch.tensor(output_pts, dtype=torch.float64), (output_pts.shape[0], 2)),
    labels=['u1', 'u2']
)

## Governing Physical Model

We consider a 3D domain $\Omega \subseteq \mathbb{R}^3$ representing a **column geometry**, with boundary $\Gamma = \partial \Omega$.  
The physical phenomenon is described by the following differenzial problem:
$$
\begin{equation}
    \begin{cases}
        u_t \left(x, y, z, t \right) - \Delta u \left( x, y, z, t \right) + F \left( t \right) & \Omega \times \left[ 0,1 \right] \\
        u_b = B \left( x, y, z, t \right) & \partial \Omega \times \left[0, 1 \right] \\
        u_0 \left( x, y, z \right) = I \left( x, y, z \right) & \Omega
    \end{cases}
    \tag{1}
\end{equation}
$$
where
\begin{equation}
    F \left( t \right) = \left(
        \begin{array}{c}
            \lambda e^{\lambda t} \\
            \lambda e^{\lambda t} - 2 \left( \alpha + \beta + 1 \right)
        \end{array}
    \right), \qquad \Omega \times \left[ 0,1 \right] ,
    \tag{2}
\end{equation}

\begin{equation}
    B \left( x, y, z, t \right) = \left(
        \begin{array}{c}
            e^{\lambda t} + \alpha x + \beta y + z \\
            e^{\lambda t} + \alpha x^2 + \beta y^2 + z^2 
        \end{array}
    \right), \qquad \partial \Omega \times \left[ 0,1 \right] ,
    \tag{3}
\end{equation}

\begin{equation}
    I \left( x, y, z \right) = \left(
        \begin{array}{c}
            1 + \alpha x + \beta y + z \\
            1 + \alpha x^2 + \beta y^2 + z^2
        \end{array}
    \right), \qquad \Omega .
    \tag{4}
\end{equation}

The unknown parameter vector is

\begin{equation}
\mu = (\lambda, \alpha, \beta). \tag{5}
\end{equation}

### Inverse Problem Formulation

The inverse problem consists of identifying the parameter vector $ \mu $ from a set of observed data.

Let $ \mathcal{D} = \{(x_i, u_i)\}_{i=1}^{N_d} $ denote a set of measurements collected on the boundary $ \partial\Omega $, where the values $ u_i $ are generated synthetically and emulate IoT sensor observations.

The goal is to recover $ \mu $ such that the reconstructed solution:
- satisfies the governing PDE,
- matches the observed data,
- respects the boundary conditions.

In the next block, the problem is defined.

In [ ]:
class ColumnParabolic(SpatialProblem, TimeDependentProblem, InverseProblem):

    input_variables = ['x','y','z','t']
    output_variables = ['u1','u2']
    spatial_domain = column_int.spatial_domain
    temporal_domain = column_int.temporal_domain
    # Definiamo il range per i parametri
    unknown_parameter_domain = CartesianDomain(
        {
            'lambda' : [0, 1],
            'alpha' : [0, 1],
            'beta' : [0, 1]
        }
    )

    @staticmethod
    def residual_u1(input_, output_, params_):
        u_t = grad(output_, input_, components=['u1'], d=['t'])
        lap_u = laplacian(output_=output_, input_=input_, components=['u1'], d=['x', 'y', 'z'])
        force_term = params_['lambda']*torch.exp(params_['lambda']*input_.extract('t'))
        return u_t - lap_u - force_term
    
    @staticmethod
    def residual_u2(input_, output_, params_):
        u_t = grad(output_, input_, components=['u2'], d=['t'])
        lap_u = laplacian(output_=output_, input_=input_, components=['u2'], d=['x', 'y', 'z'])
        force_term = params_['lambda']*torch.exp(params_['lambda']*input_.extract('t')) - 2 * (params_['alpha'] + params_['beta'] + 1)
        return u_t - lap_u - force_term
    
    @staticmethod
    def boundary_u1(input_, output_, params_):
        ref = (
            torch.exp(params_['lambda'] * input_.extract('t')) + (
                params_['alpha']*input_.extract('x') + params_['beta']*input_.extract('y') + input_.extract('z')
            )
        )
        return output_.extract('u1') - ref
    
    @staticmethod
    def boundary_u2(input_, output_, params_):
        ref = (
            torch.exp(params_['lambda'] * input_.extract('t')) + (
                params_['alpha']*(input_.extract('x')**2) + params_['beta']*(input_.extract('y')**2) + input_.extract('z')**2
            )
        )
        return output_.extract('u2') - ref
    
    @staticmethod
    def initial_u1(input_, output_, params_):
        ref = (
            1. + (
                params_['alpha']*input_.extract('x') + params_['beta']*input_.extract('y') + input_.extract('z')
            )
        )
        return output_.extract('u1') - ref
    
    @staticmethod
    def initial_u2(input_, output_, params_):
        ref = (
            1. + (
                params_['alpha']*(input_.extract('x')**2) + params_['beta']*(input_.extract('y')**2) + input_.extract('z')**2
            )
        )
        return output_.extract('u2') - ref
    
    conditions = {
        'Omega' : Condition(
            location=column_int,
            equation=SystemEquation([residual_u1,residual_u2])
        ),
        'Gamma' : Condition(
            location=column_bound,
            equation=SystemEquation([boundary_u1,boundary_u2])
        ),
        'initial' : Condition(
            location=column_initial,
            equation=SystemEquation([initial_u1,initial_u2])
        ),
        'data' : Condition(
            input_points=input_pts.extract(['x','y','z','t']),
            output_points=output_pts.extract(['u1','u2'])
        )
    }
            

After defining the class, the sampling of collocation and boundary points is performed.
Following the initial sampling step, the selected points are stored in CSV files, which are subsequently reused for further processing and analysis.

In [ ]:
problem = ColumnParabolic()

try:    
    df_omega = pd.read_csv(load_collocation_points, sep=";", index_col=0)
    df_gamma = pd.read_csv(load_boundary_points, sep=";", index_col=0)
    df_initial = pd.read_csv(load_initial_points, sep=";", index_col=0)

    problem.discretise_domain(
        1,
        locations=["Omega", "Gamma", "initial"]
    )
    problem.input_pts["Omega"] = LabelTensor(
        torch.tensor(df_omega.values),
        labels=["t", "x", "y", "z"]
    )
    problem.input_pts["Gamma"] = LabelTensor(
        torch.tensor(df_gamma.values),
        labels=["t", "x", "y", "z"]
    )
    problem.input_pts["initial"] = LabelTensor(
        torch.tensor(df_initial.values),
        labels=["t", "x", "y", "z"]
    )
except:
    problem.discretise_domain(
        n=1000,
        mode='random',
        locations=['Omega']
    )

    problem.discretise_domain(
        n=400,
        mode='random',
        locations=['Gamma', 'initial']
    )

    df_omega = pd.DataFrame(
        problem.input_pts['Omega'].tensor.detach().numpy(),
        columns=['t','x','y','z']
    )

    df_gamma = pd.DataFrame(
        problem.input_pts['Gamma'].tensor.detach().numpy(),
        columns=['t','x','y','z']
    )

    df_initial = pd.DataFrame(
        problem.input_pts['initial'].tensor.detach().numpy(),
        columns=['t','x','y','z']
    )

    df_omega.to_csv(load_collocation_points, sep=";")
    df_gamma.to_csv(load_boundary_points, sep=";")
    df_initial.to_csv(load_initial_points, sep=";")

## Physics-Informed Neural Network (PINN)

A Physics-Informed Neural Network is employed to approximate the solution

\begin{equation}
u(x,y,z;\boldsymbol{\mu}) \approx u_{\theta}(x,y,z),
\end{equation}

where $ u_{\theta} $ is a neural network parameterized by weights $ \theta $.

In the inverse setting, the unknown physical parameters $ \boldsymbol{\mu} $ are treated as **trainable variables** and optimized jointly with the network weights.

### PINN Loss Function

The training of the PINN is driven by a composite loss function:

\begin{equation}
\mathcal{L} =
\mathcal{L}_{\text{PDE}} +
\mathcal{L}_{\text{BC}} +
\mathcal{L}_{\text{IC}} +
\mathcal{L}_{\text{data}},
\end{equation}

where:

- **PDE residual loss**

\begin{equation}
\mathcal{L}_{\text{PDE}} =
\frac{1}{N_{\Omega}}
\sum_{i=1}^{N_{\Omega}}
\left\|
\Delta u_{\theta}(x_i) + (\alpha^2 + \beta^2)\pi^2 \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Boundary condition loss**

\begin{equation}
\mathcal{L}_{\text{BC}} =
\frac{1}{N_{\partial\Omega}}
\sum_{i=1}^{N_{\partial\Omega}}
\left\|
u_{\theta}(x_i) - \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Initial condition loss**

\begin{equation}
\mathcal{L}_{\text{IC}} =
\frac{1}{N_{\text{IC}}}
\sum_{i=1}^{N_{\text{IC}}}
\left\|
u_{\theta}(x_i) - \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Data loss**

\begin{equation}
\mathcal{L}_{\text{data}} =
\frac{1}{N_d}
\sum_{i=1}^{N_d}
\left\|
u_{\theta}(x_i) - u_i
\right\|^2.
\end{equation}

In [ ]:
class HardMLP(torch.nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__()
        self.layers = ResidualFeedForward(*args, **kwargs)

    # Nel metodo forward implementiamo il vincolo rigido
    def forward(self, x):
        return  self.layers(x)

Definition of the class to save the parameters during the training phase.

In [ ]:
# Directory to save the updating of parameters during the training
tmp_dir = ABS_PATH + "column_parabolic_inverse"

class SaveParameters(Callback):
    """
    Callback per salvare i parametri del modello ogni 100 epoche.
    """
    def on_train_epoch_end(self, trainer, _):
        if trainer.current_epoch % 100 == 99:
            torch.save(
                trainer.solver.problem.unknown_parameters,
                '{}/parameters_epoch{}'.format(tmp_dir, trainer.current_epoch)
            )

Inizialization of the NN model, the PINN solver, the Trainer. Then, the training phase starts.

In [ ]:
# Model
model = HardMLP(
    input_dimensions=ipt_var,
    output_dimensions=out_var,
    n_layers=lay,
    inner_size=neur
)
# Solver
pinn=RBAPINN(
    problem=problem,
    model=model,
    optimizer_kwargs={
        'lr' : lear_rate,
        'weight_decay' : decay_rt
    },
)

# Trainer
trainer=Trainer(
    solver=pinn,
    max_epochs=epochs,
    batch_size=None,
    accelerator=acc_str,
    precision='64-true',
    callbacks=[SaveParameters(), MetricTracker(), StochasticWeightAveraging(swa_lrs=swa_lr)]
)

# Training phase
trainer.train()

Plot of the losses related to the training process.

In [ ]:
my_pl = Plotter()

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Omega_loss'],
    label='Omega_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Gamma_loss'],
    label='Gamma_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['data_loss'],
    label='data_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['initial_loss'],
    label='initial_loss',
    logy=True
)

plt.savefig(loss_fig, transparent=True)
plt.show()

## Convergence Monitoring

During training, the evolution of the inferred parameters $ \mu = (\lambda, \alpha, \beta) $ is monitored and compared against the reference values used to generate the synthetic data.

This analysis provides a direct assessment of the accuracy and stability of the inverse identification process.

In [ ]:
epochs_saved = range(99, epochs, 100)
parameters = torch.empty(
    size=(int(epochs/100), 3)
)

for i, epoch in enumerate(epochs_saved):
    params_torch = torch.load('{}/parameters_epoch{}'.format(tmp_dir, epoch))
    for e, var in enumerate(pinn.problem.unknown_variables):
        parameters[i, e] = params_torch[var].data

pred_alpha, pred_beta, pred_lambda = parameters[-1, :]

# Grafico dei parametri
plt.close()
plt.plot(epochs_saved, parameters[:, 2], label='lambda', marker='o')
plt.plot(epochs_saved, parameters[:, 0], label='alpha', marker='s')
plt.plot(epochs_saved, parameters[:, 1], label='beta', marker='^')
plt.ylim(-0.1, 1)
plt.grid()
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Parameters")
plt.savefig(param_fig, transparent=True)
plt.show()

Relative errors w.r.t. the exact parameters.

In [ ]:
par_lambda = torch.tensor(.1)
par_alpha = torch.tensor(.2)
par_beta = torch.tensor(.5)

err_rel_lambda = torch.norm(pred_lambda-par_lambda)/torch.norm(par_lambda)
err_rel_alpha = torch.norm(pred_alpha-par_alpha)/torch.norm(par_alpha)
err_rel_beta = torch.norm(pred_beta-par_beta)/torch.norm(par_beta)

print("RELATIVE ERRORS")
print(f"lambda: {err_rel_lambda.item(): .2e}")
print(f"alpha: {err_rel_alpha.item(): .2e}")
print(f"beta: {err_rel_beta.item(): .2e}")

Saving the parameters and the model.

In [ ]:
df = pd.DataFrame(
    data=[[pred_lambda.item()], [pred_alpha.item()], [pred_beta.item()]],
    columns=["predictions"],
    index=["lambda", "alpha", "beta"]
)

df.to_csv(params_save, sep=";")

In [ ]:
torch.save(model, ABS_PATH + f"models/model_L{lay}_N{neur}_EP{epochs}.pth")

## Inverse Stage Outputs

At the end of the inverse stage, the following results are obtained:

- estimated physical parameters $ \mu = (\lambda, \alpha, \beta) $,
- trained PINN model consistent with physics and data,
- convergence histories for parameters and loss terms.

The inferred parameters are subsequently used in the **online stage** to compute fast reduced-order simulations.